<b>Note: </b>This model is for global population forecasting (all countries in one variable), so not for each country because the aim is to look at the large scale. Of course, the model is trained using a tree-based algorithm (Random Forest regressor, XGBoost, and LightGBM).

1. Import library

In [7]:
import os
import joblib
import warnings
import pandas as pd
import numpy as np

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from sklearn.model_selection import GridSearchCV

warnings.filterwarnings('ignore')

2. Load Data & sort values

In [2]:
df = pd.read_excel('clean_population.xlsx')
df = df.copy()
df = df.sort_values(['country', 'year'])

df.head()

,country,year,iso_code,population
0,Afghanistan,1974,AFG,12469127
1,Afghanistan,1975,AFG,12773966
2,Afghanistan,1976,AFG,13059861
3,Afghanistan,1977,AFG,13340758
4,Afghanistan,1978,AFG,13611444


3. Feature Engineering (LAG + ROLLING) with the last 3 years period

In [3]:
df['lag_1'] = df.groupby('country')['population'].shift(1)

df['lag_2'] = df.groupby('country')['population'].shift(2)

df['lag_3'] = df.groupby('country')['population'].shift(3)

# Rolling Mean
df['rolling_mean_3'] = (
    df.groupby('country')['population']
    .transform(
        lambda x:
        x.shift(1).rolling(3).mean()
    )
)

# Growth Rate
df['growth_rate'] = (
    df.groupby('country')['population']
    .pct_change()
)

# Remove Missing
df = df.dropna().reset_index(drop=True)

4. Encode Country

In [4]:
encoder = LabelEncoder()

df['country_encoded'] = encoder.fit_transform(
    df['country']
)

5. Train Test Split

In [5]:
train_df = df[df['year'] <= 2022]

test_df = df[df['year'] > 2022]

FEATURES = [
    'year',
    'country_encoded',
    'lag_1',
    'lag_2',
    'lag_3',
    'rolling_mean_3',
    'growth_rate'
]

TARGET = 'population'

X_train = train_df[FEATURES]

y_train = train_df[TARGET]

X_test = test_df[FEATURES]

y_test = test_df[TARGET]

6. Define Models & param Grid

In [6]:
model_configs = {
    'RandomForest': {
        'model': RandomForestRegressor(
            random_state=42,
            n_jobs=-1
        ),
        'params': {
            'n_estimators': [100, 300, 500],
            'max_depth': [8, 12, None],
            'min_samples_leaf': [1, 3, 5]
        }
    },
    'XGBoost': {
        'model': XGBRegressor(
            random_state=42
        ),
        'params': {
            'n_estimators': [100, 300, 500],
            'learning_rate': [0.05, 0.1, 0.2],
            'max_depth': [8, 12, None]
        }
    },
    'LightGBM': {
        'model': LGBMRegressor(
            random_state=42
        ),
        'params': {
            'n_estimators': [100, 300, 500],
            'learning_rate': [0.05, 0.1, 0.2],
            'max_depth': [8, 12, None]
        }
    }
}

7. Train Models

In [8]:
evaluation_results = []

best_model = None
best_model_name = None
best_r2 = -999999

for model_name, config in model_configs.items():
    print(f'\nGRID SEARCH: {model_name}')

    # GridSearch
    grid_search = GridSearchCV(
        estimator=config['model'],
        param_grid=config['params'],
        cv=3,
        scoring='r2',
        verbose=1,
        n_jobs=-1
    )

    # Train
    grid_search.fit(X_train, y_train)

    # Best Model
    model = grid_search.best_estimator_

    print('\nBest Params:')
    print(grid_search.best_params_)

    # Predict
    pred_test = model.predict(X_test)

    # Evaluation
    mae = mean_absolute_error(
        y_test,
        pred_test
    )

    rmse = np.sqrt(
        mean_squared_error(
            y_test,
            pred_test
        )
    )

    r2 = r2_score(
        y_test,
        pred_test
    )

    print(f'\nMAE  : {mae:.4f}')
    print(f'RMSE : {rmse:.4f}')
    print(f'R2   : {r2:.4f}')

    evaluation_results.append({
        'Model': model_name,
        'MAE': mae,
        'RMSE': rmse,
        'R2': r2,
        'Best_Params': str(
            grid_search.best_params_
        )
    })

    # Save Best Global Model
    if r2 > best_r2:
        best_r2 = r2
        best_model = model
        best_model_name = model_name


GRID SEARCH: RandomForest
Fitting 3 folds for each of 27 candidates, totalling 81 fits

Best Params:
{'max_depth': None, 'min_samples_leaf': 1, 'n_estimators': 300}

MAE  : 374950.4545
RMSE : 1834379.4842
R2   : 0.9999

GRID SEARCH: XGBoost
Fitting 3 folds for each of 27 candidates, totalling 81 fits

Best Params:
{'learning_rate': 0.1, 'max_depth': None, 'n_estimators': 500}

MAE  : 570839.1990
RMSE : 2958306.6699
R2   : 0.9997

GRID SEARCH: LightGBM
Fitting 3 folds for each of 27 candidates, totalling 81 fits
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0,000358 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1542
[LightGBM] [Info] Number of data points in the train set: 9153, number of used features: 7
[LightGBM] [Info] Start training from score 30435746,368076
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, b

8. Forecast 2025 - 2026

In [ ]:
future_results = []

countries = df['country'].unique()

for country in countries:

    country_df = (
        df[df['country'] == country]
        .sort_values('year')
        .copy()
    )

    # Check minimum 3 data
    if len(country_df) < 3:
        continue

    lag_1 = country_df.iloc[-1]['population']
    lag_2 = country_df.iloc[-2]['population']
    lag_3 = country_df.iloc[-3]['population']

    country_encoded = encoder.transform(
        [country]
    )[0]

    current_lag_1 = lag_1
    current_lag_2 = lag_2
    current_lag_3 = lag_3

    for future_year in [2025, 2026]:
        rolling_mean_3 = np.mean([
            current_lag_1,
            current_lag_2,
            current_lag_3
        ])

        growth_rate = (
            (current_lag_1 - current_lag_2)
            / current_lag_2
        )

        X_future = pd.DataFrame({
            'year': [future_year],
            'country_encoded': [
                country_encoded
            ],
            'lag_1': [
                current_lag_1
            ],
            'lag_2': [
                current_lag_2
            ],
            'lag_3': [
                current_lag_3
            ],
            'rolling_mean_3': [
                rolling_mean_3
            ],
            'growth_rate': [
                growth_rate
            ]
        })
        prediction = best_model.predict(
            X_future
        )[0]

        # Avoid Negativity
        prediction = max(
            prediction,
            0
        )

        future_results.append({
            'country': country,
            'year': future_year,
            'predicted_population': round(
                prediction
            )
        })

        # Recursive Update
        current_lag_3 = current_lag_2
        current_lag_2 = current_lag_1
        current_lag_1 = prediction

9. Save Forecast & Models

In [11]:
# Save Forecast
forecast_df = pd.DataFrame(
    future_results
)

os.makedirs('Forecast', exist_ok=True)

forecast_df.to_excel(
    'Forecast/global_population_forecast_2025_2026.xlsx',
    index=False
)

print('\nForecast saved successfully')
print(forecast_df.head())

# Save all Models
os.makedirs('models', exist_ok=True)

for model_name, config in model_configs.items():
    model = config['model']

    model.fit(
        df[FEATURES],
        df[TARGET]
    )

    # Save Model
    model_path = (
        f'models/{model_name}_population.pkl'
    )
    joblib.dump(
        model,
        model_path
    )
    print(f'{model_name} model saved')

# Save Encoder
encoder_path = 'models/country_encoder.pkl'

joblib.dump(
    encoder,
    encoder_path
)

print('Encoder saved successfully')


Forecast saved successfully
       country  year  predicted_population
0  Afghanistan  2025              42386249
1  Afghanistan  2026              43232349
2      Albania  2025               2826791
3      Albania  2026               2842920
4      Algeria  2025              47287215
RandomForest model saved
XGBoost model saved
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0,000301 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1544
[LightGBM] [Info] Number of data points in the train set: 9496, number of used features: 7
[LightGBM] [Info] Start training from score 30967207,001685
LightGBM model saved
Encoder saved successfully
